# CONFIRM 7 — Reproducing (or debunking) the published Fig. 2 error bars

Purpose: the error bars currently embedded in `paper_figures_standalone.ipynb`
for Fig. 2 (`SIG_MK`, `SIG_REL`) do **not** match what `RelCal.fit_network()`
and a properly gauge-projected MK covariance give (checked in CONFIRM_6).
This notebook tests concrete, falsifiable hypotheses for what recipe *did*
produce the published numbers, so we can pin down which is correct before
touching the figure.

Two hypotheses, motivated by inspecting the published values directly:

**MK side.** `SIG_MK` is nonzero at the reference map (143A: 0.12 deg) --
impossible for a properly common-mode-removed quantity, since that should be
exactly zero at the reference. Hypothesis: `SIG_MK` is the **raw marginal**
MK chain standard deviation, `std(alpha_i)`, with no gauge projection at all.

**Relative-estimator side.** Comparing `SIG_REL` map-by-map to the direct
*pairwise* sigma of the (map, 143A) edge from Table 1: the ratio clusters at
`~1.9--2.3` for every one of the 7 non-reference maps (mean ~2.07). Hypothesis:
`SIG_REL[m]` is the **direct pairwise sigma to the reference map, divided by
2** -- not the full GLS network marginal from `fit_network()`.

This notebook computes all four candidate quantities against the real
pipeline and reports which (if any) reproduce the published numbers to
sub-percent precision. Prints a `RESULT-7` block; copy it back verbatim.


In [1]:
import os
for v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[v]='1'
%cd /global/homes/l/lonappan/workspace/cosmic_birefringence
import numpy as np
from cosmic_bire import CBlike
from cosmic_bire.relcal import RelCal
from cosmic_bire.analytic_cov import MKResponse, closure_covariance

config = 'configs/planck_hfi.yml'
mask   = '0'
REF    = '143A'   # paper's fiducial gauge
DEG    = 180/np.pi

# --- published Fig. 2 values (from paper_figures_standalone.ipynb) ---
PUBLISHED_SIG_MK  = {"100A":0.14, "143A":0.12, "217A":0.11, "353A":0.11,
                      "100B":0.13, "143B":0.12, "217B":0.11, "353B":0.11}
PUBLISHED_SIG_REL = {"100A":0.057, "143A":0.000, "217A":0.033, "353A":0.032,
                      "100B":0.053, "143B":0.037, "217B":0.033, "353B":0.032}


/global/u2/l/lonappan/workspace/cosmic_birefringence


In [2]:
# --- MK chain (mask 0, dust-EB on) ---
lh = CBlike(config)
lh.config['likelihood']['mask'] = mask
lh.precompute()
_ = lh.getdist_samples()
samples = lh.samples
n_A = int(lh.config['likelihood'].get('number_of_A', 4))
nmaps = 2*len(lh.config['spectra']['frequencies'])

beta_mean = float(samples[:, 0].mean())
alpha_mk_mean = samples[:, 1:1+nmaps].mean(0)
labels = [l.replace('alpha_', '') for l in lh.parameter_names[1:1+nmaps]]  # 'alpha_143A' -> '143A'
K = nmaps
ref_idx = labels.index(REF)

# --- RelCal network + MK response ---
rc  = RelCal(config, mask=mask)
mkr = MKResponse(lh.analysis,
                 np.concatenate(([beta_mean], alpha_mk_mean,
                                 samples[:, 1+nmaps:1+nmaps+n_A].mean(0) if n_A else [])),
                 n_A=n_A)


Using existing /global/homes/l/lonappan/pscratch/CBDATA/theory/beam_corrected_lcdm_spectra_100_143_217_353.npy
Using existing /global/homes/l/lonappan/pscratch/CBDATA/theory/psi_l_sigma15.npy
Number of alphas: 8
Using hfi/lfi ps mask with CO. Index: 0
Importing observed cl. Filename: /global/homes/l/lonappan/pscratch/CBDATA/spectra/raw/cl_mask_percent_0_freq_100_143_217_353.npy
Importing beam smoothed LCDM cl. Filename: /global/homes/l/lonappan/pscratch/CBDATA/theory/beam_corrected_lcdm_spectra_100_143_217_353.npy
Loading /global/homes/l/lonappan/pscratch/CBDATA/theory/psi_l_sigma15.npy
Loading saved covariance matrix. Filename: /global/homes/l/lonappan/pscratch/CBDATA/covariance/cov_bin_cl_hfi_mask_percent_0_100_143_217_353_lmin51_lmax1491.npy
Loaded 218368 samples directly from /global/homes/l/lonappan/pscratch/CBDATA/chains/planck_hfi_mask_0.h5
Removed no burn in


In [3]:
# =============== MK-side candidates ===============
# Candidate A: raw marginal chain std, NO gauge projection (the suspected match)
sig_mk_raw = samples[:, 1:1+nmaps].std(0, ddof=1)

# Candidate B: properly gauge-projected differential std (what CONFIRM_6 used)
alpha_mk_cov = np.cov(samples[:, 1:1+nmaps].T)
Pm = np.eye(K); Pm[:, ref_idx] -= 1.0
sig_mk_projected = np.sqrt(np.diag(Pm @ alpha_mk_cov @ Pm.T))

print(f'{"map":6s} {"raw_marginal":>13s} {"projected":>11s} {"published":>11s} '
      f'{"raw/pub":>8s} {"proj/pub":>9s}')
for i, lbl in enumerate(labels):
    pub = PUBLISHED_SIG_MK[lbl]
    r_raw = sig_mk_raw[i]/pub if pub > 0 else float("nan")
    r_proj = sig_mk_projected[i]/pub if pub > 0 else float("nan")
    print(f'{lbl:6s} {sig_mk_raw[i]:13.4f} {sig_mk_projected[i]:11.4f} {pub:11.4f} '
          f'{r_raw:8.3f} {r_proj:9.3f}')


map     raw_marginal   projected   published  raw/pub  proj/pub
100A          0.1355      0.0890      0.1400    0.968     0.636
143A          0.1162      0.0000      0.1200    0.968     0.000
217A          0.1105      0.0529      0.1100    1.004     0.481
353A          0.1083      0.0524      0.1100    0.985     0.476
100B          0.1318      0.0819      0.1300    1.014     0.630
143B          0.1166      0.0594      0.1200    0.971     0.495
217B          0.1099      0.0531      0.1100    0.999     0.483
353B          0.1093      0.0542      0.1100    0.994     0.493


In [4]:
# =============== Relative-estimator-side candidates ===============
# Candidate A: full GLS network marginal from fit_network() (what CONFIRM_6 used)
a_rel, c_rel, labels_net = rc.fit_network(ref=REF)
sig_rel_network = np.sqrt(np.diag(c_rel))

# Candidate B: direct pairwise sigma of the (map, REF) edge, at face value
pairs = rc.fit_all_pairs()   # dict[(label_i, label_j)] -> (dalpha, sigma)
def pair_sigma_to_ref(lbl, ref=REF):
    if lbl == ref:
        return 0.0
    for (li, lj), (_, sig) in pairs.items():
        if {li, lj} == {lbl, ref}:
            return sig
    raise KeyError(f'no direct edge between {lbl} and {ref}')

sig_rel_edge = np.array([pair_sigma_to_ref(lbl) for lbl in labels_net])

# Candidate C: direct pairwise sigma to REF, divided by 2 (the suspected match)
sig_rel_edge_half = sig_rel_edge / 2.0

print(f'{"map":6s} {"network":>9s} {"edge":>8s} {"edge/2":>8s} {"published":>10s} '
      f'{"net/pub":>8s} {"edge/2/pub":>10s}')
for i, lbl in enumerate(labels_net):
    pub = PUBLISHED_SIG_REL[lbl]
    r_net = sig_rel_network[i]/pub if pub > 0 else float("nan")
    r_half = sig_rel_edge_half[i]/pub if pub > 0 else float("nan")
    print(f'{lbl:6s} {sig_rel_network[i]:9.4f} {sig_rel_edge[i]:8.4f} '
          f'{sig_rel_edge_half[i]:8.4f} {pub:10.4f} {r_net:8.3f} {r_half:10.3f}')


map      network     edge   edge/2  published  net/pub edge/2/pub
100A      0.0968   0.1199   0.0600     0.0570    1.699      1.052
143A      0.0194   0.0000   0.0000     0.0000      nan        nan
217A      0.0592   0.0650   0.0325     0.0330    1.795      0.985
353A      0.0593   0.0704   0.0352     0.0320    1.853      1.100
100B      0.0893   0.1088   0.0544     0.0530    1.684      1.027
143B      0.0651   0.0712   0.0356     0.0370    1.759      0.962
217B      0.0592   0.0650   0.0325     0.0330    1.795      0.984
353B      0.0605   0.0734   0.0367     0.0320    1.892      1.147


In [5]:
# =============== verdict: which candidate matches published to <1%? ===============
def rms_frac_residual(candidate, published_dict, labels_):
    num, den = [], []
    for i, lbl in enumerate(labels_):
        pub = published_dict[lbl]
        if pub == 0:
            continue
        num.append((candidate[i]-pub)/pub)
    return float(np.sqrt(np.mean(np.square(num))))

print('--- MK side: RMS fractional residual vs published SIG_MK ---')
print(f'  raw marginal std        : {rms_frac_residual(sig_mk_raw, PUBLISHED_SIG_MK, labels)*100:.2f}%')
print(f'  Pm-projected (gauge-correct): {rms_frac_residual(sig_mk_projected, PUBLISHED_SIG_MK, labels)*100:.2f}%')
print()
print('--- Relative-estimator side: RMS fractional residual vs published SIG_REL ---')
print(f'  fit_network() GLS marginal : {rms_frac_residual(sig_rel_network, PUBLISHED_SIG_REL, labels_net)*100:.2f}%')
print(f'  direct edge-to-ref sigma   : {rms_frac_residual(sig_rel_edge, PUBLISHED_SIG_REL, labels_net)*100:.2f}%')
print(f'  direct edge-to-ref sigma/2 : {rms_frac_residual(sig_rel_edge_half, PUBLISHED_SIG_REL, labels_net)*100:.2f}%')


--- MK side: RMS fractional residual vs published SIG_MK ---
  raw marginal std        : 2.04%
  Pm-projected (gauge-correct): 56.93%

--- Relative-estimator side: RMS fractional residual vs published SIG_REL ---
  fit_network() GLS marginal : 78.57%
  direct edge-to-ref sigma   : 108.08%
  direct edge-to-ref sigma/2 : 7.26%


In [6]:
print('==================== RESULT-7 ====================')
print('MK_SIDE (label, raw_marginal, projected, published):')
for i, lbl in enumerate(labels):
    print(f'  {lbl}: raw={sig_mk_raw[i]:.4f} proj={sig_mk_projected[i]:.4f} '
          f'pub={PUBLISHED_SIG_MK[lbl]:.4f}')
print()
print('REL_SIDE (label, network, edge, edge/2, published):')
for i, lbl in enumerate(labels_net):
    print(f'  {lbl}: network={sig_rel_network[i]:.4f} edge={sig_rel_edge[i]:.4f} '
          f'edge_half={sig_rel_edge_half[i]:.4f} pub={PUBLISHED_SIG_REL[lbl]:.4f}')
print()
print(f'RMS_RESID_MK_RAW: {rms_frac_residual(sig_mk_raw, PUBLISHED_SIG_MK, labels)*100:.2f}%')
print(f'RMS_RESID_MK_PROJECTED: {rms_frac_residual(sig_mk_projected, PUBLISHED_SIG_MK, labels)*100:.2f}%')
print(f'RMS_RESID_REL_NETWORK: {rms_frac_residual(sig_rel_network, PUBLISHED_SIG_REL, labels_net)*100:.2f}%')
print(f'RMS_RESID_REL_EDGE: {rms_frac_residual(sig_rel_edge, PUBLISHED_SIG_REL, labels_net)*100:.2f}%')
print(f'RMS_RESID_REL_EDGE_HALF: {rms_frac_residual(sig_rel_edge_half, PUBLISHED_SIG_REL, labels_net)*100:.2f}%')
print('=================================================')


==================== RESULT-7 ====================
MK_SIDE (label, raw_marginal, projected, published):
  100A: raw=0.1355 proj=0.0890 pub=0.1400
  143A: raw=0.1162 proj=0.0000 pub=0.1200
  217A: raw=0.1105 proj=0.0529 pub=0.1100
  353A: raw=0.1083 proj=0.0524 pub=0.1100
  100B: raw=0.1318 proj=0.0819 pub=0.1300
  143B: raw=0.1166 proj=0.0594 pub=0.1200
  217B: raw=0.1099 proj=0.0531 pub=0.1100
  353B: raw=0.1093 proj=0.0542 pub=0.1100

REL_SIDE (label, network, edge, edge/2, published):
  100A: network=0.0968 edge=0.1199 edge_half=0.0600 pub=0.0570
  143A: network=0.0194 edge=0.0000 edge_half=0.0000 pub=0.0000
  217A: network=0.0592 edge=0.0650 edge_half=0.0325 pub=0.0330
  353A: network=0.0593 edge=0.0704 edge_half=0.0352 pub=0.0320
  100B: network=0.0893 edge=0.1088 edge_half=0.0544 pub=0.0530
  143B: network=0.0651 edge=0.0712 edge_half=0.0356 pub=0.0370
  217B: network=0.0592 edge=0.0650 edge_half=0.0325 pub=0.0330
  353B: network=0.0605 edge=0.0734 edge_half=0.0367 pub=0.0320

RM